In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
import numpy as np

from src.config import DATA_DIR, RESULT_DIR

load_dotenv()

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

local_dir = RESULT_DIR / "implicite_segmentation"
local_dir.mkdir(parents=True, exist_ok=True)


import pysbd

_seg = pysbd.Segmenter(language="en", clean=False)  # instancier une seule fois, hors boucle

LABEL_CATS = ["Question", "Neither", "Assertion"]
SCORE_CATS = [1.0, 2.0, 3.0, 4.0, 5.0]


def segment_text(review_text: str) -> list[str]:
    return _seg.segment(review_text)

def block_to_features(block: pd.DataFrame) -> np.ndarray:
    n_b = len(block)  # dénominateur UNIQUE partagé par les deux sous-vecteurs

    # Bloc-1 : proportions des labels sur le bloc entier.
    # reindex force les 3 catégories dans l'ordre fixe ; fill_value=0
    # gère un label absent du bloc sans lever d'erreur.
    label_counts = (block["rule_based_SA-label"]
                    .value_counts()
                    .reindex(LABEL_CATS, fill_value=0))
    h1 = label_counts.to_numpy() / n_b

    # Bloc-2 : proportions des valeurs de score, MÊME dénominateur n_b.
    # Les None (lignes non-Assertion) sont ignorés par value_counts,
    # donc la somme de h2 vaut mécaniquement p_assert.
    score_counts = (block["rounded_prediction"]
                    .value_counts()
                    .reindex(SCORE_CATS, fill_value=0))
    h2 = score_counts.to_numpy() / n_b

    return np.concatenate([h1, h2])

def df_to_feature_matrix(df: pd.DataFrame):
    """Retourne (X, ids) : X de forme (n_blocs, 8), ids l'ordre des review-id."""
    ids, rows = [], []
    # sort=False : préserve l'ordre d'apparition, pas de tri implicite
    for rid, block in df.groupby("review-id", sort=False):
        ids.append(rid)
        rows.append(block_to_features(block))
    X = np.vstack(rows)            # (n_blocs, 8) — vstack garantit la forme 2D
    return X, np.array(ids)


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
output_file_path_sentence = "sentences_degrees_calibration.csv"
df = pd.read_csv(local_dir / output_file_path_sentence, encoding="utf-8-sig")


In [ ]:
df

In [ ]:

hwt_df = df[df["label"] == 0]
mgt_df = df[df["label"] == 1]

# df_to_feature_matrix renvoie un TUPLE (X, ids) — il faut déballer.
X_hwt, ids_hwt = df_to_feature_matrix(hwt_df)
X_mgt, ids_mgt = df_to_feature_matrix(mgt_df)

# Labels constants par source. np.full fixe dtype=int et la bonne longueur.
y_hwt = np.full(len(X_hwt), 0, dtype=int)   # HWT → 0
y_mgt = np.full(len(X_mgt), 1, dtype=int)   # MGT → 1

# Dataset supervisé
X = np.vstack([X_hwt, X_mgt])         # (n_hwt + n_mgt, 8)
y = np.concatenate([y_hwt, y_mgt])

In [ ]:
X

In [ ]:
print(len(df))
print(len(hwt_df['review-id'].unique()))
print(len(mgt_df['review-id'].unique()))
print(len(df['review-id'].unique()))
df

## Récupération de la base de test

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

output_file_path_sentence = "sentences_degrees_test.csv"
df_test = pd.read_csv(local_dir / output_file_path_sentence, encoding="utf-8-sig")

hwt_df_test = df_test[df_test["label"] == 0]
mgt_df_test = df_test[df_test["label"] == 1]

# df_to_feature_matrix renvoie un TUPLE (X, ids) — il faut déballer.
X_hwt_test, ids_hwt_test = df_to_feature_matrix(hwt_df_test)
X_mgt_test, ids_mgt_test = df_to_feature_matrix(mgt_df_test)

# Labels constants par source. np.full fixe dtype=int et la bonne longueur.
y_hwt_test = np.full(len(X_hwt_test), 0, dtype=int)   # HWT → 0
y_mgt_test = np.full(len(X_mgt_test), 1, dtype=int)   # MGT → 1

# Dataset supervisé
X_test = np.vstack([X_hwt_test, X_mgt_test])         # (n_hwt + n_mgt, 8)
y_test = np.concatenate([y_hwt_test, y_mgt_test])

In [ ]:
print(len(df_test))
print(len(hwt_df_test['review-id'].unique()))
print(len(mgt_df_test['review-id'].unique()))
print(len(df_test['review-id'].unique()))
df_test

# Entrainement

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X_tr = X
y_tr = y
X_te = X_test
y_te = y_test
# 2) STANDARDISATION fit sur le TRAIN seul, puis transform du test.
scaler = StandardScaler().fit(X_tr)      # paramètres estimés UNIQUEMENT sur train
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)          # test transformé avec stats du train -> pas de fuite

# 3) MLP : deux petites couches, early stopping pour éviter le surapprentissage.
clf = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation="relu",
    solver="adam",
    alpha=1e-4,                  # régularisation L2
    early_stopping=True,
    n_iter_no_change=50,
    max_iter=5000,
    random_state=42,
)
clf.fit(X_tr_s, y_tr)

# 4) ÉVALUATION honnête sur le test.
y_pred = clf.predict(X_te_s)
y_proba = clf.predict_proba(X_te_s)[:, 1]
print("itérations effectuées :", clf.n_iter_)
print("\nmatrice de confusion [[TN FP][FN TP]] :")
print(confusion_matrix(y_te, y_pred))
print("\nclassification report :")
print(classification_report(y_te, y_pred, digits=3))
print("AUC :", round(roc_auc_score(y_te, y_proba), 4))

result_file = f"mlp_result_1.npz"
np.savez(local_dir / f"{result_file}", y_true=y_te, y_proba=y_proba)



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

# from_estimator recalcule predict_proba en interne et trace directement.
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(clf, X_te_s, y_te, ax=ax, name="MLP baseline")

# Diagonale de référence : classifieur aléatoire (AUC = 0.5).
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="aléatoire")

ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbe ROC — MLP baseline")
ax.legend()
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()